read input image

In [1]:
import os, re, cv2, torch, pandas as pd
import torchvision.models as models
import torchvision.transforms as transforms
from pathlib import Path


image_folder = Path(r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_fold2")

read label

In [2]:
label_path  = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\scientificProject\data\labels.csv"
label_df = pd.read_csv(label_path)
label_df["name"] = label_df["name"].astype(str)
label = {}
for item in range(len(label_df['name'])):
    label[int(label_df['name'][item].split('.')[0])] = label_df['grade'][item]
    

In [3]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class CustomImageDataset(Dataset):
    def __init__(self, image_folder, label_dict, transform=None):
        self.image_folder = Path(image_folder)  # 🔧 اینجا اصلاح شد
        self.label_dict = label_dict
        self.transform = transform

        self.existing_ids = []
        for i in range(1, 713):
            img_path = self.image_folder / f"{i}.png"
            if img_path.exists():
                self.existing_ids.append(i)

    def __len__(self):
        return len(self.existing_ids)

    def __getitem__(self, idx):
        image_id = self.existing_ids[idx]
        img_path = self.image_folder / f"{image_id}.png"
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = self.label_dict[image_id]
        return image, label
    
    
train_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_fold2"
test_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\classification_level1\AND_result_test"
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ساخت دیتاست آموزش
train_dataset = CustomImageDataset(train_dir, label, transform=transform)

# ساخت دیتاست تست
test_dataset = CustomImageDataset(test_dir, label, transform=transform)

In [45]:
# pip install timm

In [46]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm

# استفاده از GPU در صورت وجود
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# دیتالودرها
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# مدل ConvNeXt-Tiny
model = timm.create_model("convnext_tiny", pretrained=True, num_classes=5)
model.to(device)

# تابع هزینه و بهینه‌ساز
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# آموزش مدل
epochs = 10
for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100 * correct / total
    print(f"\nEpoch {epoch+1}: Train Loss = {train_loss:.4f}, Train Acc = {train_acc:.2f}%")

# ارزیابی روی داده‌های تست
model.eval()
test_loss = 0
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

test_acc = 100 * correct / total
print(f"\nTest Loss = {test_loss:.4f}, Test Accuracy = {test_acc:.2f}%")


In [48]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import timm
from tqdm import tqdm
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# هایپرپارامترهای قابل تست
param_grid = {
    'lr': [1e-3, 1e-4],
    'weight_decay': [0, 1e-4, 1e-2],
    'optimizer': ['adam', 'adamw'],
    'batch_size': [16, 32]
}

# ساخت تمام ترکیب‌ها (حدود 2x3x2x2 = 24 حالت)
param_combinations = list(itertools.product(
    param_grid['lr'],
    param_grid['weight_decay'],
    param_grid['optimizer'],
    param_grid['batch_size']
))

results = []
best_acc = 0
best_params = None

# تعداد ایپاک
epochs = 10

for idx, (lr, wd, opt_name, batch_size) in enumerate(param_combinations):
    print(f"\n🔍 Testing combination {idx+1}/{len(param_combinations)} - lr={lr}, wd={wd}, opt={opt_name}, batch={batch_size}")

    # دیتالودرها
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # مدل جدید برای هر اجرا
    model = timm.create_model("convnext_tiny", pretrained=True, num_classes=5)
    model.to(device)

    # loss و optimizer
    criterion = nn.CrossEntropyLoss()
    if opt_name == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    # آموزش
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # ارزیابی
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total
    print(f"✅ Test Accuracy: {acc:.2f}%")

    results.append(((lr, wd, opt_name, batch_size), acc))
    if acc > best_acc:
        best_acc = acc
        best_params = (lr, wd, opt_name, batch_size)

# نمایش بهترین حالت
print("\n🏆 Best Hyperparameters:")
print(f"Learning Rate: {best_params[0]}")
print(f"Weight Decay: {best_params[1]}")
print(f"Optimizer: {best_params[2]}")
print(f"Batch Size: {best_params[3]}")
print(f"Test Accuracy: {best_acc:.2f}%")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
import random
from tqdm import tqdm
import itertools

# --- Augmentations ---
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- datasets ---
train_dataset.transform = train_transform
test_dataset.transform = test_transform

# --- Search space ---
lr_list = [5e-5, 1e-4, 2e-4]
wd_list = [1e-5, 1e-4, 5e-4]
opt_list = ['adamw', 'sgd']
bs_list = [16, 32, 64]

param_combinations = list(itertools.product(lr_list, wd_list, opt_list, bs_list))
random.shuffle(param_combinations)
param_combinations = param_combinations[:20]  # 20 ترکیب تصادفی

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Training Loop ---
epochs = 10
best_acc = 0
best_params = None

for idx, (lr, wd, opt_name, batch_size) in enumerate(param_combinations):
    print(f"\n🧪 Config {idx+1}/20 - lr={lr}, wd={wd}, opt={opt_name}, bs={batch_size}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = timm.create_model("convnext_tiny", pretrained=True, num_classes=5)
    model.to(device)

    criterion = nn.CrossEntropyLoss()

    if opt_name == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd, momentum=0.9)

    # آموزش
    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # ارزیابی
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    acc = 100 * correct / total
    print(f"✅ Accuracy: {acc:.2f}%")

    if acc > best_acc:
        best_acc = acc
        best_params = (lr, wd, opt_name, batch_size)

# --- Best result ---
print("\n🏆 Best Hyperparameters:")
print(f"Learning Rate: {best_params[0]}")
print(f"Weight Decay: {best_params[1]}")
print(f"Optimizer: {best_params[2]}")
print(f"Batch Size: {best_params[3]}")
print(f"Test Accuracy: {best_acc:.2f}%")
